# Imports and Global Setup

In [1]:
pip_dir = "/kaggle/input/tabpfn-2-0-1-offline/"
!pip install --no-index --find-links=$pip_dir tabpfn 

Looking in links: /kaggle/input/tabpfn-2-0-1-offline/
Processing /kaggle/input/tabpfn-2-0-1-offline/tabpfn-2.0.1-py3-none-any.whl
Processing /kaggle/input/tabpfn-2-0-1-offline/nvidia_cuda_nvrtc_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl (from torch>=2.1->tabpfn)
Processing /kaggle/input/tabpfn-2-0-1-offline/nvidia_cuda_runtime_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl (from torch>=2.1->tabpfn)
Processing /kaggle/input/tabpfn-2-0-1-offline/nvidia_cuda_cupti_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl (from torch>=2.1->tabpfn)
Processing /kaggle/input/tabpfn-2-0-1-offline/nvidia_cudnn_cu12-9.1.0.70-py3-none-manylinux2014_x86_64.whl (from torch>=2.1->tabpfn)
Processing /kaggle/input/tabpfn-2-0-1-offline/nvidia_cublas_cu12-12.4.5.8-py3-none-manylinux2014_x86_64.whl (from torch>=2.1->tabpfn)
Processing /kaggle/input/tabpfn-2-0-1-offline/nvidia_cufft_cu12-11.2.1.3-py3-none-manylinux2014_x86_64.whl (from torch>=2.1->tabpfn)
Processing /kaggle/input/tabpfn-2-0-1-offline/nvidia_c

In [2]:
import os
import torch
import warnings
import numpy as np
import pandas as pd
from tabpfn import TabPFNRegressor
from sklearn.model_selection import KFold, GroupKFold
from sklearn.metrics import mean_squared_error

# Suppress all warnings
warnings.filterwarnings("ignore")

# Set a seed for reproducibility
np.random.seed(42)
torch.manual_seed(42)

TARGET = 'HOMELESS_RATE'

# Determine the device to use (GPU if available, otherwise CPU)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

OFFLINE_MODEL_PATH = "/kaggle/input/tabpfn-2-0-1-offline/tabpfn-v2-regressor.ckpt"

Using device: cuda


# Data Loading 

In [3]:
train_file = "/kaggle/input/california-homelessness-prediction-challenge/train.csv"
test_file = "/kaggle/input/california-homelessness-prediction-challenge/test.csv"

train_df = pd.read_csv(train_file)
test_df = pd.read_csv(test_file)
submission_df = pd.read_csv("/kaggle/input/california-homelessness-prediction-challenge/sample_submission.csv")

features = [col for col in train_df.columns if col != TARGET and col != 'ID']

X = train_df[features].values
y = train_df[TARGET].values
X_test = test_df[features].values

In [4]:
submission_df

,ID,HOMELESS_RATE
0,AL_13,1.23
1,LA_10,1.23
2,SD_15,1.23
3,SB_11,1.23
4,SB_09,1.23
5,OC_28,1.23
6,AL_02,1.23
7,RV_29,1.23
8,SB_01,1.23
9,SC_14,1.23


# Model Training

In [5]:
# IDをアルファベットと、_ 数字に分けます

train_df_id_split = train_df['ID'].str.split('_', expand=True)
train_df['STATE'] = train_df_id_split[0]
train_df.drop('ID', axis = 1, inplace = True)
train_df

,HOMELESS_RATE,AGE_U18_PCT,AGE_18_24_PCT,AGE_25_34_PCT,AGE_35_44_PCT,AGE_45_54_PCT,AGE_55_59_PCT,AGE_60_61_PCT,AGE_62_64_PCT,AGE_65_69_PCT,...,DISABILITY_POP_PCT,NODISABILITY_POP_PCT,TOTAL_HOUSEHOLDS_PCT,FAMILY_HH_TOTAL,FAMILY_HH_CHILD_LT18_PCT,NONFAMILY_SINGLE_MALE_PCT,NONFAMILY_SINGLE_FEMALE_PCT,MULTI_PERSON_NONFAMILY_HH_PCT,INDIVIDUALS_NOT_IN_FAMILY_UNITS_PCT,STATE
0,0.000000,0.388210,0.054130,0.058164,0.061302,0.161269,0.062087,0.037431,0.051104,0.201053,...,0.460607,0.020397,0.305727,0.251373,0.076544,0.174829,0.054354,0.021966,0.054354,SC
1,0.000999,0.368737,0.089002,0.132744,0.137555,0.156465,0.082081,0.031715,0.035453,0.083043,...,0.457923,0.023203,0.227111,0.184183,0.064947,0.119236,0.042928,0.016283,0.042928,RV
2,0.004736,0.429991,0.071720,0.204073,0.176398,0.127522,0.046399,0.020886,0.025102,0.067669,...,0.517711,0.030974,0.237949,0.198297,0.094440,0.103857,0.039651,0.014248,0.039651,SC
3,0.001269,0.476342,0.058899,0.115471,0.156583,0.147944,0.064417,0.027537,0.032103,0.082117,...,0.498431,0.023747,0.272830,0.234998,0.118750,0.116247,0.037832,0.013240,0.037832,AL
4,0.000394,0.513634,0.078841,0.112881,0.145494,0.148405,0.066102,0.024644,0.029827,0.098252,...,0.485158,0.029264,0.249969,0.189581,0.093464,0.096118,0.060387,0.016156,0.060387,SAC
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
125,0.002657,0.409175,0.089357,0.166851,0.145895,0.128258,0.069670,0.029918,0.035483,0.090299,...,0.510729,0.031529,0.222504,0.154082,0.065876,0.088206,0.068421,0.021257,0.068421,AL
126,0.002084,0.540632,0.103225,0.136034,0.126443,0.120192,0.054043,0.027022,0.043790,0.092542,...,0.497139,0.031751,0.216901,0.130743,0.059997,0.070746,0.086158,0.022954,0.086158,RV
127,0.000396,0.262039,0.029467,0.072036,0.073173,0.083160,0.052606,0.033818,0.040789,0.164244,...,0.415653,0.011322,0.243746,0.206764,0.050035,0.156729,0.036982,0.015426,0.036982,OC
128,0.033858,0.357389,0.103374,0.202590,0.149430,0.125208,0.055579,0.022972,0.030457,0.083999,...,0.494994,0.027024,0.187579,0.116812,0.042014,0.074798,0.070767,0.025516,0.070767,LA


In [6]:
test_df_id_split = test_df['ID'].str.split('_', expand=True)
test_df['STATE'] = test_df_id_split[0]
test_df.drop('ID', axis = 1, inplace =True)
test_df

,AGE_U18_PCT,AGE_18_24_PCT,AGE_25_34_PCT,AGE_35_44_PCT,AGE_45_54_PCT,AGE_55_59_PCT,AGE_60_61_PCT,AGE_62_64_PCT,AGE_65_69_PCT,AGE_70_79_PCT,...,DISABILITY_POP_PCT,NODISABILITY_POP_PCT,TOTAL_HOUSEHOLDS_PCT,FAMILY_HH_TOTAL,FAMILY_HH_CHILD_LT18_PCT,NONFAMILY_SINGLE_MALE_PCT,NONFAMILY_SINGLE_FEMALE_PCT,MULTI_PERSON_NONFAMILY_HH_PCT,INDIVIDUALS_NOT_IN_FAMILY_UNITS_PCT,STATE
0,0.342169,0.088348,0.147178,0.134551,0.144855,0.073641,0.032186,0.036751,0.097467,0.157331,...,0.494866,0.019192,0.228051,0.161873,0.062935,0.098939,0.066178,0.022780,0.066178,AL
1,0.370642,0.090580,0.197082,0.152119,0.127954,0.060101,0.026158,0.032274,0.088054,0.103299,...,0.496151,0.025846,0.211684,0.121713,0.046176,0.075537,0.089971,0.030425,0.089971,LA
2,0.462069,0.074806,0.135474,0.148376,0.121019,0.064783,0.028337,0.037049,0.110785,0.140659,...,0.484905,0.040998,0.250991,0.191846,0.085025,0.106820,0.059145,0.018705,0.059145,SD
3,0.324003,0.070298,0.124383,0.123496,0.137429,0.089677,0.036605,0.056111,0.136795,0.175807,...,0.465484,0.023813,0.274098,0.193920,0.053325,0.140595,0.080177,0.022799,0.080177,SB
4,0.443576,0.091267,0.133863,0.124290,0.142770,0.074939,0.034057,0.045485,0.097654,0.117705,...,0.510769,0.028775,0.267414,0.216761,0.088704,0.128057,0.050654,0.020307,0.050654,SB
5,0.345233,0.081234,0.104020,0.105479,0.137022,0.087862,0.038239,0.062192,0.143134,0.202819,...,0.484538,0.017342,0.280499,0.231649,0.074469,0.157180,0.048850,0.010474,0.048850,OC
6,0.487742,0.064962,0.143756,0.170670,0.127228,0.062915,0.020173,0.028112,0.097568,0.122834,...,0.475009,0.033005,0.258501,0.203825,0.111050,0.092775,0.054676,0.010586,0.054676,AL
7,0.498786,0.086012,0.125515,0.126715,0.125975,0.062975,0.026369,0.035735,0.098126,0.151396,...,0.492707,0.030368,0.235513,0.184078,0.076372,0.107706,0.051436,0.016932,0.051436,RV
8,0.702073,0.099037,0.142570,0.136040,0.106274,0.044893,0.018610,0.023997,0.075148,0.052185,...,0.486777,0.053110,0.207542,0.116613,0.062714,0.053899,0.090929,0.024895,0.090929,SB
9,0.406645,0.049874,0.050471,0.116922,0.165461,0.086682,0.032312,0.044992,0.128407,0.203779,...,0.489639,0.010642,0.296291,0.272478,0.115306,0.157172,0.023813,0.006041,0.023813,SC


In [7]:
%%time

test_predictions = []
oof_predictions = np.zeros(len(X))

n_splits = 10

gkf = GroupKFold(n_splits=n_splits)

print(f"Starting {n_splits}-fold cross-validation...")

for fold, (train_index, val_index) in enumerate(gkf.split(X, y, groups=train_df['STATE'])):

    X_train, X_val = X[train_index], X[val_index]
    y_train, y_val = y[train_index], y[val_index]

    model = TabPFNRegressor(device=device, ignore_pretraining_limits=True, model_path=OFFLINE_MODEL_PATH)
    model.fit(X_train,y_train)

    val_preds = model.predict(X_val)
    oof_predictions[val_index] = val_preds
    mse = mean_squared_error(y_val, val_preds)

    print(f"MSE score on validation set for Fold {fold+1}: {mse}")

    test_preds = model.predict(X_test)
    test_predictions.append(test_preds)


# oof_df = pd.DataFrame({'ID': train_df['ID'], TARGET: oof_predictions})
# oof_df.to_csv('oof.csv', index=False)

submission_df[TARGET] = np.mean(test_predictions, axis=0)

submission_df.to_csv('submission.csv', index=False)
submission_df.head()


Starting 10-fold cross-validation...
MSE score on validation set for Fold 1: 6.773067864422496e-06
MSE score on validation set for Fold 2: 1.2205203763285029e-05
MSE score on validation set for Fold 3: 8.635678225078815e-06
MSE score on validation set for Fold 4: 1.0800655481663945e-05
MSE score on validation set for Fold 5: 2.6386453279543855e-05
MSE score on validation set for Fold 6: 0.00029492688815358295
MSE score on validation set for Fold 7: 8.031199708009469e-05
MSE score on validation set for Fold 8: 9.141616923929884e-06
MSE score on validation set for Fold 9: 5.2170017777878836e-05
MSE score on validation set for Fold 10: 1.750996934610539e-05
CPU times: user 14.9 s, sys: 843 ms, total: 15.8 s
Wall time: 15.7 s


,ID,HOMELESS_RATE
0,AL_13,0.003558
1,LA_10,0.012699
2,SD_15,0.001488
3,SB_11,0.002642
4,SB_09,0.001163
